# Summarize train/test/full cohorts


## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import col_config_og, col_config_sensitivity
import numpy as np
import pandas as pd
import src.clean as clean

pd.set_option("display.max_columns", 500)

In [ ]:
def merge_dfs(processed, raw):
    merged = pd.merge(processed, raw, on="DAWBA_ID", how="left")
    if "Participant_Type2_x" in merged.columns:
        merged = merged.drop(columns=["Participant_Type2_x"]).rename(
            columns={"Participant_Type2_y": "Participant_Type2"}
        )
    return merged

In [ ]:
# df_raw = pd.read_csv(
#     BASE_PATH / "data/complete_raw/dawba_raw.csv"
# ).drop(index=234)
# df_raw[(df_raw['DAWBA_ID']==234875) | df_raw['Participant_Type2'].isna()]['Participant_Type2']

In [ ]:
data_imp_path = BASE_PATH / "data/cleaned/processed"
## Train
train_p = pd.read_parquet(data_imp_path / "parent.parquet")
train_s = pd.read_parquet(data_imp_path / "self.parquet")
# Test
test_p = pd.read_parquet(data_imp_path / "test/parent.parquet")
test_s = pd.read_parquet(data_imp_path / "test/self.parquet")

## Raw data
df_raw = pd.read_csv(
    BASE_PATH
    / "data/complete_raw/comppsych_dawbas_20210414-0.csv"  # this has been manually cleaned to fill in errenous entries
    # BASE_PATH / "data/complete_raw/dawba_raw.csv" #this is completely raw
).drop(index=234)
df_raw_sub = df_raw[
    [
        "DAWBA_ID",
        "SEX",
        "Eligible",
        "DAWBA_Age_at_visit",
        "SDAN",
        "PLUSID",
        "Clinical_Visit_Date",
        "Overall_date",
        "p_dawba_sdq_date",
        "s_dawba_sdq_date",
        "Participant_Type2",
    ]
].copy()
df_raw_sub = df_raw_sub.replace(
    {
        "SEX": {"MALE": "Male", "FEMALE": "Female"},
        "Eligible": {
            "Excluded: meets exclusionary criteria (substance use, psychosis, etc.)": "Inelligible: Meets exclusion criteria",
            "Excluded: does not meet criteria": "Inellgible: Does not meet inclusion criteria",
            "Include: can't scan (braces, etc.)": "Eligible",
            "On hold: contact again after specified amount of time": "Eligible",
            "On hold: low priority": "Eligible",
            "Include": "Eligible",
            "Excluded: cannot be reached or scheduled, all contact options exhausted": "Eligible",
        },
    }
)
df_raw_sub["Eligible"] = df_raw_sub["Eligible"].fillna("Eligible")


df_merge_train_p = merge_dfs(train_p, df_raw_sub)
df_merge_test_p = merge_dfs(test_p, df_raw_sub)
df_merge_train_s = merge_dfs(train_s, df_raw_sub)
df_merge_test_s = merge_dfs(test_s, df_raw_sub)

## Get Ages

In [ ]:
# r = pd.concat([
#     df_merge_train_p[['PLUSID', 'SDAN', 'DAWBA_ID', 'Overall_date', 'Clinical_Visit_Date', 's_dawba_sdq_date', 'p_dawba_sdq_date', 'DAWBA_Age_at_visit']],
# 	df_merge_test_p[['PLUSID', 'SDAN', 'DAWBA_ID', 'Overall_date', 'Clinical_Visit_Date', 's_dawba_sdq_date', 'p_dawba_sdq_date', 'DAWBA_Age_at_visit']],
# 	df_merge_train_s[['PLUSID', 'SDAN', 'DAWBA_ID', 'Overall_date', 'Clinical_Visit_Date', 's_dawba_sdq_date', 'p_dawba_sdq_date', 'DAWBA_Age_at_visit']],
# 	df_merge_test_s[['PLUSID', 'SDAN', 'DAWBA_ID', 'Overall_date', 'Clinical_Visit_Date', 's_dawba_sdq_date', 'p_dawba_sdq_date', 'DAWBA_Age_at_visit']]
# ])
# r['train_parent'] = np.where(r['DAWBA_ID'].isin(df_merge_train_p['DAWBA_ID']), 1, 0)
# r['test_parent'] = np.where(r['DAWBA_ID'].isin(df_merge_test_p['DAWBA_ID']), 1, 0)
# r['train_self'] = np.where(r['DAWBA_ID'].isin(df_merge_train_s['DAWBA_ID']), 1, 0)
# r['test_self'] = np.where(r['DAWBA_ID'].isin(df_merge_test_s['DAWBA_ID']), 1, 0)
# r.drop_duplicates(subset=['PLUSID', 'SDAN', 'DAWBA_ID'], inplace=True)

# exp_path = BASE_PATH / "data/get_age_dir/get_age.csv"
# r.reset_index(drop=True).to_csv(exp_path)

In [ ]:
age_df = pd.read_csv(BASE_PATH / "data/get_age_dir/age_stats.csv").set_index("group")
age_df

## Extract

In [ ]:
from src.utils import col_config_sensitivity

COUNT_COLS = ["SEX", "Eligible", "Participant_Type2"]
from src.summarize import summarize_text, get_col

In [ ]:
from functools import reduce

all_entries = {}
for count_col in COUNT_COLS:
    all_entries[count_col] = set(df_merge_train_p[count_col].unique())
    all_entries[count_col].update(set(df_merge_test_p[count_col].unique()))
    all_entries[count_col].update(set(df_merge_train_s[count_col].unique()))
    all_entries[count_col].update(set(df_merge_test_s[count_col].unique()))
all_entries


train_s_summary = get_col(
    df_merge=df_merge_train_s,
    set_type="train",
    impact_type="self",
    count_col_entries=all_entries,
    text_cols=col_config_sensitivity["self_text"],
    quest_cols=col_config_sensitivity["self_quest"],
)
test_s_summary = get_col(
    df_merge=df_merge_test_s,
    set_type="test",
    impact_type="self",
    count_col_entries=all_entries,
    text_cols=col_config_sensitivity["self_text"],
    quest_cols=col_config_sensitivity["self_quest"],
)

s_summary_merge = pd.merge(train_s_summary, test_s_summary, on="Feature", how="inner")

train_p_summary = get_col(
    df_merge=df_merge_train_p,
    set_type="train",
    impact_type="parent",
    count_col_entries=all_entries,
    text_cols=col_config_sensitivity["parent_text"],
    quest_cols=col_config_sensitivity["parent_quest"],
)
test_p_summary = get_col(
    df_merge=df_merge_test_p,
    set_type="test",
    impact_type="parent",
    count_col_entries=all_entries,
    text_cols=col_config_sensitivity["parent_text"],
    quest_cols=col_config_sensitivity["parent_quest"],
)

dfs = [train_s_summary, test_s_summary, train_p_summary, test_p_summary]
combined_summary_table = reduce(
    lambda left, right: pd.merge(left, right, on="Feature", how="inner"),
    dfs,
)

question_impact_rows = [
    "Questions Responded To Avg (std)",
    "Total Words in entire response Avg (std) [25%, 50%, 75%]",
    # "Impact Score Avg [25%, 50%, 75%]",
    # "Quest Sum Score Avg [25%, 50%, 75%]"
]

question_impact_table = combined_summary_table.loc[question_impact_rows]
gen_summary_table = combined_summary_table.drop(index=question_impact_rows)
## Add age
age_col = "Age Avg [25%, 50%, 75%]"
gen_summary_table.loc[age_col, "self_train (n=103)"] = (
    f"{age_df.loc["train_self", "mean"]:.2f} [{age_df.loc["train_self", "25%"]:.2f}, {age_df.loc["train_self", "50%"]:.2f}, {age_df.loc["train_self", "75%"]:.2f}]"
)
gen_summary_table.loc[age_col, "self_test (n=50)"] = (
    f"{age_df.loc["test_self", "mean"]:.2f} [{age_df.loc["test_self", "25%"]:.2f}, {age_df.loc["test_self", "50%"]:.2f}, {age_df.loc["test_self", "75%"]:.2f}]"
)
gen_summary_table.loc[age_col, "parent_train (n=71)"] = (
    f"{age_df.loc["train_parent", "mean"]:.2f} [{age_df.loc["train_parent", "25%"]:.2f}, {age_df.loc["train_parent", "50%"]:.2f}, {age_df.loc["train_parent", "75%"]:.2f}]"
)
gen_summary_table.loc[age_col, "parent_test (n=35)"] = (
    f"{age_df.loc["test_parent", "mean"]:.2f} [{age_df.loc["test_parent", "25%"]:.2f}, {age_df.loc["test_parent", "50%"]:.2f}, {age_df.loc["test_parent", "75%"]:.2f}]"
)


display(gen_summary_table)
display(question_impact_table)

## Get distributions

In [ ]:
text_length_list_train_s = summarize_text(
    text_cols=col_config_sensitivity["self_text"],
    df=df_merge_train_s,
    col_name="",
    return_lists=True,
)
text_length_list_train_p = summarize_text(
    text_cols=col_config_sensitivity["parent_text"],
    df=df_merge_train_p,
    col_name="",
    return_lists=True,
)
text_length_list_test_s = summarize_text(
    text_cols=col_config_sensitivity["self_text"],
    df=df_merge_test_s,
    col_name="",
    return_lists=True,
)
text_length_list_test_p = summarize_text(
    text_cols=col_config_sensitivity["parent_text"],
    df=df_merge_test_p,
    col_name="",
    return_lists=True,
)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(text_length_list_train_s, bins=30, label="Train Self")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.title("Distribution of Text Lengths-Train Self")
plt.legend()
plt.show()

In [ ]:
plt.hist(text_length_list_train_p, bins=30, label="Train Parent")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.title("Distribution of Text Lengths-Train Parent")
plt.legend()
plt.show()

In [ ]:
plt.hist(text_length_list_test_s, bins=30, label="Test Self")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.title("Distribution of Text Lengths-Test Self")
plt.legend()
plt.show()

In [ ]:
plt.hist(text_length_list_test_p, bins=30, label="Test Parent")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.title("Distribution of Text Lengths-Test Parent")
plt.legend()
plt.show()